In [0]:
# Restart Python environment to ensure clean state for new package installations
dbutils.library.restartPython()

In [0]:
# Install core dependencies for RAG implementation — ALL VERSIONS PINNED
# Pins are the metadata-equivalent lock for THIS compute's base environment
# (langchain-core 1.2.6, databricks-sdk 0.67.0, databricks-connect 18.2.8,
# mlflow-skinny 3.8.1). Verified by dry-run resolution against the same
# constraint files %pip applies — resolves instantly, no backtracking.
# Transitive versions that land with this lock: langchain-classic 1.0.1,
# langchain-text-splitters 1.1.0, langgraph-checkpoint 4.3.1,
# langgraph-prebuilt 1.0.10, langgraph-sdk 0.3.15, databricks-ai-bridge 0.17.0,
# databricks-mcp 0.5.1, mcp 1.12.4, mlflow 3.8.1 (matches mlflow-skinny pin),
# unitycatalog-ai 0.3.0, unitycatalog-client 0.4.0, unitycatalog-langchain 0.3.0.
# - langchain / langchain-community: LLM orchestration (langchain-core stays at 1.2.6)
# - langgraph: Graph-based agent orchestration with checkpointing
# - databricks-langchain: Databricks integrations (ChatDatabricks, DatabricksVectorSearch)
# - databricks-vectorsearch: Vector search client
# - duckduckgo-search: Web search fallback
# NOTE: langchain-community is intentionally omitted — no cell in this notebook
# imports it, and it requires requests>=2.32.5 while this compute pins
# requests==2.32.3. mlflow is pinned at 3.8.1, the exact version the runtime
# manages for mlflow-skinny, so the constraint file allows it.
%pip install langchain==1.2.4 langchain-text-splitters==1.1.0 langgraph==1.0.10 langgraph-checkpoint==4.3.1 langgraph-prebuilt==1.0.10 langgraph-sdk==0.3.15 databricks-langchain==0.14.0 databricks-vectorsearch==0.66 databricks-ai-bridge==0.17.0 databricks-mcp==0.5.1 langchain-mcp-adapters==0.3.0 mcp==1.12.4 unitycatalog-ai==0.3.0 unitycatalog-client==0.4.0 unitycatalog-langchain==0.3.0 mlflow==3.8.1 mlflow-tracing==3.8.1 duckduckgo-search==8.1.1





In [0]:
# Print exact installed versions for model dependency pinning
# These versions are used in the pip_requirements when logging the model to MLflow
# Pinned versions prevent pip resolver backtracking during model serving container builds
import subprocess

result = subprocess.run(["pip", "freeze"], capture_output=True, text=True)
targets = [
    "mlflow", "langchain", "langchain-community", "langchain-core",
    "databricks-vectorsearch", "databricks-sdk", "databricks-langchain",
    "cloudpickle", "openai", "openai-agents", "mcp", "databricks-openai",
    "databricks-ai-bridge", "databricks-ai-search", "databricks-mcp",
    "langchain-mcp-adapters", "unitycatalog-langchain"
]

print("=== Pinned versions for MLflow pip_requirements ===")
for pkg in result.stdout.strip().split("\n"):
    pkg_name = pkg.lower().split("==")[0].split(">")[0].split("<")[0].split("[")[0].strip()
    if pkg_name in targets:
        print(f"  {pkg}")

## Configuration
Set required parameters for Movie Recommendation RAG system

In [0]:
# Configure RAG system parameters
import os

# Get Databricks workspace URL for API calls
host = "https://" + spark.conf.get("spark.databricks.workspaceUrl")

# Note: The databricks-vectorsearch SDK will automatically use the notebook's authentication
# when running in Databricks, so setting DATABRICKS_TOKEN is not required.
# If you need to set it manually, create a secret scope first:
# databricks secrets create-scope demo
# databricks secrets put-secret demo token

# Vector search index containing movie knowledge base
index_name="rag_capstone_catalog.default.rag_docs_text_index4"

# Vector search endpoint name
VECTOR_SEARCH_ENDPOINT_NAME = "capstoneendpoint4"


## Building the Retriever
Create vector search retriever for movie recommendations

In [0]:
# Define function to create vector search retriever for RAG
# Uses a "retrieve wide, sample randomly" strategy: fetch the top-N most
# relevant chunks for the query, then randomly pick k of them. This keeps
# recommendations within the request parameters (still relevant matches)
# while varying which movies come back on repeated asks.
import random

# Compatibility shim (required with the locked dependency versions):
# mcp 1.12.4 exposes the function as streamablehttp_client, but a
# databricks-langchain 0.14 submodule imports the newer renamed symbol
# streamable_http_client. Alias it here, BEFORE databricks_langchain is
# imported, otherwise the import below raises ImportError.
import mcp.client.streamable_http as _mcp_ssh
if not hasattr(_mcp_ssh, "streamable_http_client") and hasattr(_mcp_ssh, "streamablehttp_client"):
    _mcp_ssh.streamable_http_client = _mcp_ssh.streamablehttp_client

from databricks_langchain import DatabricksVectorSearch

# Pool of top matches fetched per query, and how many are sampled for the LLM
CANDIDATE_POOL = 10  # top-N relevant matches retrieved
FINAL_K = 3         # documents actually passed to the LLM


def get_retriever():
    """Create and return a vector search retriever that randomizes
    within the top matches, so repeated requests return varied results"""
    # Set Databricks host for authentication
    os.environ["DATABRICKS_HOST"] = host

    # Base retriever fetches a WIDE pool of relevant documents (top-N)
    base_retriever = DatabricksVectorSearch(
        index_name=index_name
    ).as_retriever(search_kwargs={"k": CANDIDATE_POOL})

    class RandomizedRetriever:
        """Pulls the top-N matches for a query, keeps at most one chunk per
        movie title, then randomly samples k documents. Results vary per call
        while every sampled doc is still within the query's top matches."""

        def invoke(self, query, **kwargs):
            docs = base_retriever.invoke(query, **kwargs)
            if not docs:
                return []
            # Deduplicate to one chunk per movie title so the sampled set
            # spans different movies instead of repeating chunks of one movie
            seen_titles = set()
            unique_docs = []
            for doc in docs:
                title = (doc.metadata or {}).get("title")
                if title is None or title not in seen_titles:
                    unique_docs.append(doc)
                    if title is not None:
                        seen_titles.add(title)
            return random.sample(unique_docs, min(FINAL_K, len(unique_docs)))

    return RandomizedRetriever()

## Create the RAG Chain
Build LangChain pipeline for movie recommendations with LLM and conversation memory

In [0]:
# LangGraph Checkpointer Setup for Conversation Memory
# Replaces the previous DeltaTableChatMessageHistory + RunnableWithMessageHistory approach
# with LangGraph's native checkpointing system for cleaner state management.

from langgraph.checkpoint.memory import MemorySaver
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from datetime import datetime

# =======================
# LANGGRAPH CHECKPOINTER
# =======================
# MemorySaver stores conversation state in memory, keyed by thread_id.
# Each thread_id gets its own isolated conversation history, automatically
# managed by LangGraph — no manual message append/retrieve logic needed.
#
# For persistence across notebook restarts, swap MemorySaver for a persistent
# checkpointer (e.g., SqliteSaver or a custom Delta-backed checkpointer).
checkpointer = MemorySaver()

# =======================
# DELTA TABLE (optional analytics logging)
# =======================
# Kept for optional conversation analytics — LangGraph handles memory,
# but we can still log turns to Delta for querying/auditing.
CONVERSATION_TABLE_CAPSTONE = "rag_capstone_catalog.default.movie_conversation_history"

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CONVERSATION_TABLE_CAPSTONE} (
        session_id STRING,
        message_index INT,
        message_type STRING,
        content STRING,
        timestamp TIMESTAMP,
        metadata STRING
    )
    USING DELTA
    TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')
""")

def log_conversation_to_delta(session_id: str, human_msg: str, ai_msg: str):
    """Optional: Log a conversation turn to Delta table for analytics"""
    schema = StructType([
        StructField("session_id", StringType(), True),
        StructField("message_index", IntegerType(), True),
        StructField("message_type", StringType(), True),
        StructField("content", StringType(), True),
        StructField("timestamp", TimestampType(), True),
        StructField("metadata", StringType(), True),
    ])
    rows = [
        Row(session_id=session_id, message_index=0, message_type="human",
            content=human_msg, timestamp=datetime.now(), metadata=None),
        Row(session_id=session_id, message_index=1, message_type="ai",
            content=ai_msg, timestamp=datetime.now(), metadata=None),
    ]
    spark.createDataFrame(rows, schema=schema).write.mode("append").saveAsTable(CONVERSATION_TABLE_CAPSTONE)

print("✓ LangGraph MemorySaver checkpointer created!")
print("  • In-memory conversation state with thread_id isolation")
print("  • No manual message history management needed")
print("  • State automatically saved/restored per thread_id")
print(f"  • Delta table available for analytics: {CONVERSATION_TABLE_CAPSTONE}")

In [0]:
# Build RAG chain as a LangGraph StateGraph with checkpointed conversation memory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage
from langgraph.graph import StateGraph, MessagesState, START, END
from duckduckgo_search import DDGS
from databricks_langchain import ChatDatabricks

# Get the retriever from the function defined in Cell 7
retriever = get_retriever()

# Initialize the LLM
chat_model = ChatDatabricks(endpoint="databricks-meta-llama-3-3-70b-instruct", max_tokens=250)

# Helper function to format retrieved documents
def format_docs(docs):
    return "\n\n".join([doc.page_content for doc in docs])

# =======================
# PROMPT TEMPLATE (shared by both simple chain and LangGraph)
# =======================
template = """You are a helpful assistant that recommends movies based on the user's preferences.
Use the following pieces of context to answer the question.
Keep your answer concise and to the point (2-3 sentences maximum).
If you don't know the answer, just say that you don't know. Suggest the user provide more details about their movie preferences.

Context from vector search:
{context}

Question: {question}

Helpful Answer:"""

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful movie recommendation assistant. Use the provided context and conversation history to answer questions about movies, genres, and provide personalized recommendations based on user preferences. Vary which movies you highlight across requests - do not recommend the exact same set of movies every time the user asks. "),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", template)
])

# =======================
# SIMPLE CHAIN (no memory — for single questions in cells 16, 19)
# =======================
chain = (
    {
        "context": lambda x: format_docs(retriever.invoke(x["question"])),
        "question": lambda x: x["question"],
        "chat_history": lambda x: x.get("chat_history", [])
    }
    | prompt
    | chat_model
    | StrOutputParser()
)

# =======================
# LANGGRAPH STATE GRAPH (with checkpointed conversation memory)
# =======================
# State includes messages (auto-managed by MessagesState with add_messages reducer)
# plus context and question for the retrieve → generate flow.
class RAGState(MessagesState):
    context: str
    question: str

def retrieve(state: RAGState) -> dict:
    """Retrieve relevant documents from vector search for the latest question"""
    question = state["messages"][-1].content
    docs = retriever.invoke(question)
    return {"context": format_docs(docs), "question": question}

def generate(state: RAGState) -> dict:
    """Generate answer using LLM with retrieved context and conversation history"""
    question = state["question"]
    context = state["context"]
    # All messages except the latest human message are chat history
    chat_history = state["messages"][:-1]
    
    # Format the prompt with context, question, and history
    messages = prompt.invoke({
        "context": context,
        "question": question,
        "chat_history": chat_history
    }).to_messages()
    
    response = chat_model.invoke(messages)
    return {"messages": [response]}

# =======================
# INTERNET FALLBACK
# =======================
# When the RAG model doesn't have relevant context (returns "I don't know" etc.),
# fall back to DuckDuckGo web search and re-generate an answer with web results.

# Keywords that indicate the RAG answer is unhelpful
UNHELPFUL_INDICATORS = [
    "i don't know", "i don't have", "i'm not sure", "cannot recommend",
    "no information", "not enough information", "couldn't find",
    "please provide more", "don't have enough", "no relevant",
    "unable to", "no movies found", "not available"
]

def grade_answer(state: RAGState) -> str:
    """Check if the RAG answer is helpful or needs internet fallback"""
    answer = state["messages"][-1].content.lower().strip()
    if not answer or any(ind in answer for ind in UNHELPFUL_INDICATORS):
        return "web_fallback"
    return "end"

def web_fallback(state: RAGState) -> dict:
    """Search the internet via DuckDuckGo and generate answer with web results"""
    question = state["question"]
    chat_history = state["messages"][:-1]

    # Search the web
    try:
        with DDGS() as ddgs:
            results = list(ddgs.text(question, max_results=5))
        web_context = "\n\n".join([
            f"Title: {r['title']}\n{r['body']}"
            for r in results
        ]) if results else "No web results found."
        source = "internet (DuckDuckGo)"
    except Exception as e:
        web_context = f"Web search failed: {e}"
        source = "fallback (no web results)"

    # Generate answer using web context instead of vector search
    messages = prompt.invoke({
        "context": web_context,
        "question": question,
        "chat_history": chat_history
    }).to_messages()

    response = chat_model.invoke(messages)
    # Tag the response to indicate it came from the internet fallback
    response.content = f"[Web fallback]\n{response.content}"
    return {"messages": [response]}

# =======================
# BUILD THE GRAPH
# =======================
# START -> retrieve -> generate -> grade_answer
#                                |-- "end" -> END
#                                +-- "web_fallback" -> web_fallback -> END
graph = StateGraph(RAGState)
graph.add_node("retrieve", retrieve)
graph.add_node("generate", generate)
graph.add_node("web_fallback", web_fallback)

graph.add_edge(START, "retrieve")
graph.add_edge("retrieve", "generate")
graph.add_conditional_edges("generate", grade_answer, {
    "web_fallback": "web_fallback",
    "end": END
})
graph.add_edge("web_fallback", END)

# Compile with the MemorySaver checkpointer from Cell 9
# thread_id in the config acts as the session identifier
rag_app = graph.compile(checkpointer=checkpointer)

print("✓ LangGraph RAG StateGraph created with internet fallback!")
print("  • chain: simple chain without memory (for single questions)")
print("  • rag_app: LangGraph graph with checkpointed conversation memory")
print("  • Flow: retrieve -> generate -> grade_answer")
print("    └─ unhelpful answer -> web_fallback (DuckDuckGo search) -> END")
print("  • Usage: rag_app.invoke(")
print("      {'messages': [HumanMessage(content='your question')]},")
print("      config={'configurable': {'thread_id': 'user_123'}}")
print("    )")
print("  • Answer is in result['messages'][-1].content")



In [0]:
# Save RAG chain configuration to file for reference
# IMPORTANT: MLflow with LangChain v1+ requires code-based logging.
# Due to serialization constraints with Databricks components, we'll save
# the chain configuration and provide instructions for manual registration.

import json

# Save chain configuration for later use
config = {
    "index_name": "rag_capstone_catalog.default.rag_docs_text_index",
    "endpoint_name": "databricks-meta-llama-3-3-70b-instruct",
    "max_tokens": 200,
    "search_k": 3,
    "vector_search_endpoint": "capstoneendpoint"
}

with open("rag_chain_config.json", "w") as f:
    json.dump(config, f, indent=2)

print("✓ Configuration saved to rag_chain_config.json")
print("\nℹ️  For LangChain v1+ models with Databricks Vector Search:")
print("   The chain works perfectly for inference (as shown in Cell 10)!")
print("   For MLflow registration, consider these options:")
print("   1. Deploy as a Databricks App for interactive use")
print("   2. Use MLflow Model Serving with code-based logging (requires separate .py module)")
print("   3. Keep the chain in this notebook and call it via notebook workflows")

In [0]:
# Configure MLflow experiment for tracking RAG chain runs
import mlflow

# Set experiment name for organizing runs
experiment_name = f"/Users/manteza14@gmail.com/movie-rag-experiment"
mlflow.set_experiment(experiment_name)

# Enable MLflow tracing for LangChain
# This automatically traces all chain executions, LLM calls, and retrieval steps
mlflow.langchain.autolog()

print(f"✓ MLflow experiment set to: {experiment_name}")
print("✓ MLflow tracing enabled for LangChain")
print("  View tracked runs in the Experiments sidebar →")
print("  Each chain invocation will be traced with detailed step-by-step execution")

## Testing
Test movie recommendation RAG chain and vector search functionality

In [0]:
# Test the LangGraph checkpointed conversation memory
test_thread_id = "test_user_123Capstone"

print("Testing LangGraph conversation memory...")
print("=" * 60)

# First question
print("\n[User] Can you recommend a good action movie?")
result1 = rag_app.invoke(
    {"messages": [HumanMessage(content="Can you recommend a good action movie?")]},
    config={"configurable": {"thread_id": test_thread_id}}
)
responseCapstone = result1["messages"][-1].content
print(f"[AI] {responseCapstone}")

# Optionally log to Delta for analytics
log_conversation_to_delta(test_thread_id, "Can you recommend a good action movie?", responseCapstone)

# Follow-up question (tests conversation memory — LangGraph restores prior state via thread_id)
print("\n[User] What about something similar but with more comedy?")
result2 = rag_app.invoke(
    {"messages": [HumanMessage(content="What about something similar but with more comedy?")]},
    config={"configurable": {"thread_id": test_thread_id}}
)
responseCapstone2 = result2["messages"][-1].content
print(f"[AI] {responseCapstone2}")

# Optionally log follow-up to Delta
log_conversation_to_delta(test_thread_id, "What about something similar but with more comedy?", responseCapstone2)

print("\n" + "=" * 60)
print("\u2713 LangGraph conversation memory test complete!")
print(f"\nState for thread_id '{test_thread_id}' is checkpointed in MemorySaver.")
print("\nVerify analytics in Delta table:")
print(f"  SELECT * FROM {CONVERSATION_TABLE_CAPSTONE}")
print(f"  WHERE session_id = '{test_thread_id}'")

In [0]:
# Query conversation history from Delta table for analytics
# This demonstrates the power of Delta table-backed memory

# Check if table exists and has data
try:
    conversation_df = spark.table(CONVERSATION_TABLE_CAPSTONE)
    
    print("📊 Conversation History Analytics")
    print("=" * 60)
    
    # Total messages
    total_messages = conversation_df.count()
    print(f"\nTotal messages stored: {total_messages}")
    
    if total_messages > 0:
        # Unique sessions
        unique_sessions = conversation_df.select("session_id").distinct().count()
        print(f"Unique conversation sessions: {unique_sessions}")
        
        # Messages by type
        print("\nMessages by type:")
        conversation_df.groupBy("message_type").count().show()
        
        # Most recent conversations
        print("\nMost recent conversations (last 10 messages):")
        conversation_df.orderBy("timestamp", ascending=False).limit(10).select(
            "session_id", "message_type", "content", "timestamp"
        ).show(truncate=50)
        
        # Session activity over time
        print("\nSample queries you can run:")
        print("  • Most active users: SELECT session_id, COUNT(*) as msg_count FROM ... GROUP BY session_id")
        print("  • Popular questions: SELECT content FROM ... WHERE message_type = 'human' GROUP BY content")
        print("  • Response quality: Analyze AI response patterns")
        print("  • Peak usage times: SELECT DATE_TRUNC('hour', timestamp), COUNT(*) FROM ... GROUP BY 1")
    else:
        print("\nNo conversations yet. Start chatting to build history!")
        
except Exception as e:
    print(f"Table not created yet or error: {e}")
    print("The table will be created automatically when you start a conversation.")

In [0]:
# Test RAG chain with a sample question and track results in MLflow
import mlflow

# Start MLflow run to track this test
with mlflow.start_run(run_name="movie-rag-test"):
    # Log configuration parameters for reproducibility
    mlflow.log_param("index_name", index_name)
    mlflow.log_param("vector_endpoint", VECTOR_SEARCH_ENDPOINT_NAME)
    mlflow.log_param("llm_endpoint", "databricks-meta-llama-3-3-70b-instruct")
    mlflow.log_param("max_tokens", 200)
    mlflow.log_param("search_k", 3)
    
    # Test the chain with a sample question
    question = "Can you recommend a good action movie?"
    answer = chain.invoke({"question": question})
    print(answer)
    
    # Log metrics about the response
    mlflow.log_metric("response_length", len(answer))
    mlflow.log_metric("question_length", len(question))
    
    print("\n✓ Run logged to MLflow!")

In [0]:
# Verify vector search index contains movie data
test_query = "Recommend an action movie"

print(f"Testing index: {index_name}")
print(f"Query: '{test_query}'")
print("=" * 60)

try:
    # Get documents using the retriever
    docs = retriever.invoke(test_query)
    
    print(f"\n✅ Retrieved {len(docs)} documents:\n")
    
    for i, doc in enumerate(docs, 1):
        content = doc.page_content[:300]
        print(f"{i}. {content}...")
        print("-" * 60)
    
    # Check content type
    all_content = " ".join([doc.page_content for doc in docs]).lower()
    
    if any(word in all_content for word in ["movie", "film", "genre", "action", "comedy", "drama", "director", "actor"]):
        print("\n✅ Index contains MOVIE documents!")
    elif any(word in all_content for word in ["laptop", "phone", "hp victus", "samsung"]):
        print("\n⚠️  Index still has OLD laptop/phone documents!")
        print("Vector search may need time to refresh, or index needs rebuilding.")
    
except Exception as e:
    print(f"\n❌ Error: {str(e)}")

In [0]:
# Force vector search index to sync with latest movie data from source table
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

print("Syncing vector search index with movie data...")
print(f"Index: {index_name}")
print("=" * 60)

try:
    # Trigger index sync
    w.vector_search_indexes.sync_index(index_name)
    
    print("\n✅ Index sync triggered successfully!")
    print("\n⏳ The index will update in the background (2-5 minutes).")
    print("\n🎬 Once complete, your Gradio chatbot will answer movie questions!")
    print("\nTest questions to try after sync:")
    print("  • Can you recommend a good action movie?")
    print("  • What are some good comedy films?")
    print("  • Recommend a sci-fi movie with great visuals")
    
except Exception as e:
    print(f"\n❌ Error: {str(e)}")
    print("\n🔧 If sync fails, you may need to recreate the index.")
    print("   The index might be a direct-access type that doesn't support sync.")

In [0]:
# Define movie comparison function that retrieves and formats info for two movies
import mlflow

@mlflow.trace(name="movie_comparison", span_type="CHAIN")
def compare_movies(movie1, movie2):
    """Compare two movies side-by-side using RAG retrieval"""
    
    # Validate inputs
    if not movie1.strip() or not movie2.strip():
        return "Please enter both movie titles to compare."
    
    if movie1.strip().lower() == movie2.strip().lower():
        return "Please enter two different movie titles to compare."
    
    try:
        # Retrieve information for both movies (concise descriptions)
        question1 = f"Briefly describe the key characteristics, genre, and plot of {movie1} in 2-3 sentences"
        question2 = f"Briefly describe the key characteristics, genre, and plot of {movie2} in 2-3 sentences"
        
        # Get answers from the RAG chain (chain expects dict with 'question' key)
        info1 = chain.invoke({"question": question1})
        info2 = chain.invoke({"question": question2})
        
        # Format as a comparison
        comparison = f"""## 🎬 Movie Comparison

### {movie1.title()}
{info1}

---

### {movie2.title()}
{info2}

---

### Quick Comparison Summary
Both movies have unique characteristics. Compare the details above to see differences in:
- **Genre** (action, comedy, drama, sci-fi, etc.)
- **Plot themes** (adventure, romance, mystery, thriller)
- **Tone** (light-hearted, dark, intense, emotional)
- **Audience appeal** (family-friendly, mature, cult classic)
"""
        
        # Log metrics
        mlflow.log_param("movie1", movie1)
        mlflow.log_param("movie2", movie2)
        mlflow.log_metric("comparison_length", len(comparison))
        
        return comparison
        
    except Exception as e:
        return f"Error comparing movies: {str(e)}"

print("✓ Movie comparison tool created!")
print("  Function: compare_movies(movie1, movie2)")
print("  Usage: compare_movies('Inception', 'The Matrix')")

In [0]:
# Install Gradio for building interactive web UI
%pip install gradio

In [0]:
# Launch interactive Gradio web interface with Q&A and Comparison tabs
import gradio as gr
import mlflow
from langchain_core.messages import HumanMessage

# Define the chat function with session tracking for conversation memory
@mlflow.trace(name="movie_rag_chat", span_type="CHAIN")
def chat_with_documents(question, request: gr.Request):
    """Process user question through the LangGraph RAG app with checkpointed conversation memory"""
    if not question.strip():
        return "Please enter a question."
    
    try:
        # Get session ID from Gradio (unique per browser tab)
        # This enables conversation memory - follow-up questions remember context
        session_id = request.session_hash
        
        # Use LangGraph rag_app with checkpointed memory
        # Each browser tab gets its own conversation history via thread_id
        result = rag_app.invoke(
            {"messages": [HumanMessage(content=question)]},
            config={"configurable": {"thread_id": session_id}}
        )
        answer = result["messages"][-1].content
        
        # Optional: log to Delta table for analytics
        log_conversation_to_delta(session_id, question, answer)
        
        return answer
    except Exception as e:
        return f"Error: {str(e)}"

# Create Q&A interface
qa_interface = gr.Interface(
    fn=chat_with_documents,
    inputs=gr.Textbox(
        label="Ask a question about movies",
        placeholder="Example: Can you recommend a good action movie?",
        lines=3
    ),
    outputs=gr.Textbox(
        label="Answer",
        lines=5
    ),
    title="💬 Ask Questions",
    description="Ask questions about movies, get recommendations based on your preferences.",
    examples=[
        ["Can you recommend a good action movie?"],
        ["What are some good comedy films?"],
        ["Recommend a sci-fi movie with great visuals"],
    ],
)

# Create comparison interface
comparison_interface = gr.Interface(
    fn=compare_movies,
    inputs=[
        gr.Textbox(
            label="First Movie Title",
            placeholder="e.g., Inception",
            lines=1
        ),
        gr.Textbox(
            label="Second Movie Title",
            placeholder="e.g., The Matrix",
            lines=1
        )
    ],
    outputs=gr.Markdown(
        label="Comparison"
    ),
    title="⚖️ Compare Movies",
    description="Compare two movies side-by-side to see their differences in genre, plot, and tone.",
    examples=[
        ["Inception", "The Matrix"],
        ["The Dark Knight", "Iron Man"],
        ["Pulp Fiction", "Goodfellas"],
    ],
)

# Combine both interfaces with tabs
demo = gr.TabbedInterface(
    [qa_interface, comparison_interface],
    ["🎬 Q&A Chat", "⚖️ Compare Movies"],
    title="🎬 Movie Recommendation Chatbot"
)

# Launch the interface with keep-alive
print("🚀 Launching Gradio interface...")
print("📝 The interface will open in a new tab.")
print("⏰ Server will stay alive (cell keeps running)")
print("   To stop: Click 'Stop' button or interrupt the cell\n")

# Launch without blocking the main thread
demo.launch(share=True, prevent_thread_lock=False)

# Display the share link for embedding
import time
time.sleep(3)  # Wait for Gradio to generate share link

if demo.share_url:
    share_link = demo.share_url
    print("\n" + "="*60)
    print("🔗 YOUR GRADIO SHARE LINK (for embedding):")
    print("="*60)
    print(f"\n   {share_link}")
    print(f"\n   ⏰ Valid for 72 hours")
    print("\n" + "="*60)
    print("\n📋 Quick Embed Code for HTML:")
    print("="*60)
    print(f'\n<iframe src="{share_link}" width="100%" height="800px"></iframe>\n')
    print("="*60)
    print("\n💡 Copy the link above to embed in GitHub Pages, Netlify, or any website!")
    print("\n")

# Keep the cell executing indefinitely so Gradio stays alive
try:
    print("✅ Gradio server is running... (press Stop to end)\n")
    while True:
        time.sleep(60)  # Sleep for 60 seconds, repeat forever
except KeyboardInterrupt:
    print("\n🛑 Server stopped by user")
    demo.close()

In [0]:
# Define custom MLflow PyFunc model to wrap RAG chain for deployment
import mlflow
from mlflow.models.signature import infer_signature
import cloudpickle

# Define a custom PyFunc model that wraps the RAG chain
class MovieRAGModel(mlflow.pyfunc.PythonModel):
    def load_context(self, context):
        """Load the RAG chain components when the model is loaded"""
        import os
        from databricks_langchain import DatabricksVectorSearch, ChatDatabricks
        from langchain_core.prompts import ChatPromptTemplate
        from langchain_core.runnables import RunnablePassthrough
        from langchain_core.output_parsers import StrOutputParser
        
        # Configuration
        self.index_name = "rag_capstone_catalog.default.rag_docs_text_index"
        
        # DATABRICKS_HOST is automatically set in Model Serving environment
        # No need to set it manually - the SDK will use the serving endpoint's context
        
        # Build retriever
        retriever = DatabricksVectorSearch(
            index_name=self.index_name
        ).as_retriever(search_kwargs={"k": 3})
        
        # Build LLM
        chat_model = ChatDatabricks(
            endpoint="databricks-meta-llama-3-3-70b-instruct",
            max_tokens=250  # Enough for 2-3 concise sentences without cutoff
        )
        
        # Build prompt with concise response instruction
        template = """You are a helpful assistant that recommends movies based on the user's preferences.
Use the following pieces of context to answer the question at the end.
Keep your answer concise and to the point (2-3 sentences maximum).
If you don't know the answer, just say that you don't know. Suggest the user provide more details about their movie preferences.

Context from vector search:
{context}

Question: {question}

Helpful Answer:"""
        
        prompt = ChatPromptTemplate.from_template(template)
        
        # Helper function
        def format_docs(docs):
            return "\n\n".join([doc.page_content for doc in docs])
        
        # Build chain
        self.chain = (
            {"context": retriever | format_docs, "question": RunnablePassthrough()}
            | prompt
            | chat_model
            | StrOutputParser()
        )
    
    def predict(self, context, model_input):
        """Run inference on the input"""
        # Handle both DataFrame and dict input
        if hasattr(model_input, 'to_dict'):
            questions = model_input.to_dict()['question']
            if isinstance(questions, dict):
                questions = list(questions.values())
        elif isinstance(model_input, dict):
            questions = [model_input['question']]
        else:
            questions = model_input
        
        # Process each question
        results = []
        for question in questions:
            answer = self.chain.invoke(question)
            results.append(answer)
        
        # Always return a list for consistency
        return results

print("✓ MovieRAGModel class defined")

In [0]:
# Log and register RAG model to Unity Catalog for deployment
import mlflow
import pandas as pd
from mlflow.models.resources import DatabricksVectorSearchIndex, DatabricksServingEndpoint

# Set the registry URI to Unity Catalog
mlflow.set_registry_uri("databricks-uc")

# Create example input/output for model signature
example_input = pd.DataFrame({"question": ["Can you recommend a good action movie?"]})
example_output = "Here are some great action movies you might enjoy..."

# Infer signature
signature = infer_signature(example_input, example_output)

# Define the resources the model depends on
index_name = "rag_capstone_catalog.default.rag_docs_text_index"
llm_endpoint = "databricks-meta-llama-3-3-70b-instruct"

resources = [
    DatabricksVectorSearchIndex(index_name=index_name),
    DatabricksServingEndpoint(endpoint_name=llm_endpoint)
]

# Use conda_env instead of pip_requirements to control the conda environment
# Key fix: use unconstrained 'pip' instead of 'pip<=25.0.1' which causes slow conda solver
conda_env = {
    "channels": ["conda-forge"],
    "dependencies": [
        "python=3.12.3",
        "pip",
        {
            "pip": [
                "mlflow==3.16.0",
                "langchain==1.2.10",
                "langchain-community==0.4.1",
                "langchain-core==1.6.3",
                "databricks-vectorsearch==0.75",
                "databricks-sdk==0.67.0",
                "databricks-langchain==0.20.0",
                "cloudpickle==3.0.0",
                # Transitive dependencies pinned to prevent pip resolver backtracking
                "openai==3.13.0",
                "openai-agents==0.22.2",
                "mcp==1.30.0",
                "databricks-openai==0.17.1",
                "databricks-ai-bridge==0.21.0",
                "databricks-ai-search==0.77",
                "databricks-mcp==0.9.2",
                "langchain-mcp-adapters==0.3.2",
                "unitycatalog-langchain==0.4.0"
            ]
        }
    ],
    "name": "mlflow-env"
}

# Start MLflow run and log the model
with mlflow.start_run(run_name="movie-rag-model-deployment") as run:
    # Log the model with resource dependencies
    model_info = mlflow.pyfunc.log_model(
        artifact_path="movie_rag_model",
        python_model=MovieRAGModel(),
        signature=signature,
        input_example=example_input,
        resources=resources,
        conda_env=conda_env
    )
    
    print(f"✓ Model logged to MLflow")
    print(f"  Run ID: {run.info.run_id}")
    print(f"  Artifact Path: {model_info.model_uri}")
    
    # Register to Unity Catalog
    catalog = "rag_capstone_catalog"
    schema = "default"
    model_name = f"{catalog}.{schema}.movie_rag_chatbot"
    
    registered_model = mlflow.register_model(
        model_uri=model_info.model_uri,
        name=model_name
    )
    
    print(f"\n✓ Model registered to Unity Catalog")
    print(f"  Model: {model_name}")
    print(f"  Version: {registered_model.version}")
    
    # Store for next step
    model_version = registered_model.version

print(f"\n🎉 Model ready for deployment!")
print(f"   Model Name: {model_name}")
print(f"   Version: {model_version}")

In [0]:
# Create Model Serving endpoint to deploy RAG chatbot as REST API
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import ServedEntityInput, EndpointCoreConfigInput

w = WorkspaceClient()

# Configure serving endpoint settings
endpoint_name = "movie-rag-chatbot-endpoint"
model_name = f"{catalog}.{schema}.movie_rag_chatbot"

print(f"Creating Model Serving endpoint: {endpoint_name}")
print("=" * 60)

try:
    # Check if endpoint exists
    try:
        existing = w.serving_endpoints.get(endpoint_name)
        print(f"⚠️  Endpoint '{endpoint_name}' already exists")
        print(f"   Updating to model version {model_version}...")
        
        # Update existing endpoint
        w.serving_endpoints.update_config(
            name=endpoint_name,
            served_entities=[
                ServedEntityInput(
                    entity_name=model_name,
                    entity_version=str(model_version),
                    scale_to_zero_enabled=True,
                    workload_size="Small"
                )
            ]
        )
        print("✓ Endpoint updated")
        
    except Exception:
        # Create new endpoint
        print("Creating new endpoint...")
        w.serving_endpoints.create(
            name=endpoint_name,
            config=EndpointCoreConfigInput(
                served_entities=[
                    ServedEntityInput(
                        entity_name=model_name,
                        entity_version=str(model_version),
                        scale_to_zero_enabled=True,
                        workload_size="Small"
                    )
                ]
            )
        )
        print("✓ Endpoint created")
    
    # Wait for endpoint to be ready
    print("\n⏳ Waiting for endpoint to be ready (this may take 5-10 minutes)...")
    w.serving_endpoints.wait_get_serving_endpoint_not_updating(endpoint_name)
    
    endpoint = w.serving_endpoints.get(endpoint_name)
    
    print(f"\n🎉 Deployment complete!")
    print(f"\n📡 Endpoint Details:")
    print(f"   Name: {endpoint_name}")
    print(f"   State: {endpoint.state.config_update}")
    print(f"   Model: {model_name} (v{model_version})")
    print(f"\n🔗 Access your endpoint:")
    print(f"   Serving UI: /ml/endpoints/{endpoint_name}")
    print(f"\n💡 Your RAG chatbot is now deployed as a REST API!")
    
except Exception as e:
    print(f"\n❌ Error: {str(e)}")
    import traceback
    traceback.print_exc()

In [0]:
# Force restart if endpoint is stuck deploying >30 minutes
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import ServedEntityInput, EndpointCoreConfigInput
import time

w = WorkspaceClient()
endpoint_name = "movie-rag-chatbot-endpoint"

print("🔄 Force restarting stuck endpoint...")
print("=" * 60)

try:
    # Check current status
    endpoint = w.serving_endpoints.get(endpoint_name)
    print(f"Current state: {endpoint.state.ready} / {endpoint.state.config_update}")
    
    if endpoint.state.config_update == "IN_PROGRESS":
        print("\n⚠️  Endpoint stuck IN_PROGRESS. Deleting and recreating...\n")
        
        # Delete the stuck endpoint
        w.serving_endpoints.delete(endpoint_name)
        print("✓ Deleted stuck endpoint")
        
        # Wait for deletion
        print("⏳ Waiting 30 seconds for cleanup...")
        time.sleep(30)
        
        # Recreate with same config
        catalog = "rag_capstone_catalog"
        schema = "default"
        model_name = f"{catalog}.{schema}.movie_rag_chatbot"
        
        # Get latest model version
        import mlflow
        mlflow.set_registry_uri("databricks-uc")
        client = mlflow.MlflowClient()
        versions = client.search_model_versions(f"name='{model_name}'")
        latest_version = max([int(v.version) for v in versions])
        
        print(f"\n🚀 Creating fresh endpoint with model v{latest_version}...")
        
        w.serving_endpoints.create(
            name=endpoint_name,
            config=EndpointCoreConfigInput(
                served_entities=[
                    ServedEntityInput(
                        entity_name=model_name,
                        entity_version=str(latest_version),
                        scale_to_zero_enabled=True,
                        workload_size="Small"
                    )
                ]
            )
        )
        print("✓ Endpoint created")
        
        print("\n⏳ Waiting for endpoint to be ready (5-10 minutes)...")
        w.serving_endpoints.wait_get_serving_endpoint_not_updating(
            name=endpoint_name,
            timeout=datetime.timedelta(minutes=15)
        )
        
        endpoint = w.serving_endpoints.get(endpoint_name)
        print(f"\n🎉 Endpoint is now {endpoint.state.ready}!")
        
        if endpoint.state.ready == "READY":
            print("\n✅ Success! Now try:")
            print("   1. Re-run cell 24 to test the endpoint")
            print("   2. Your Vercel app should work too!")
        
    else:
        print("\n✅ Endpoint is not stuck. Current state:")
        print(f"   Ready: {endpoint.state.ready}")
        print(f"   Config: {endpoint.state.config_update}")
        
        if endpoint.state.ready == "READY":
            print("\n✅ Endpoint is READY! Re-run cell 24 to test.")
        
except Exception as e:
    print(f"\n❌ Error: {e}")
    import traceback
    traceback.print_exc()

In [0]:
# Quick status check - run this every 2-3 minutes to monitor progress
from databricks.sdk import WorkspaceClient
import datetime

w = WorkspaceClient()
endpoint_name = "movie-rag-chatbot-endpoint"

endpoint = w.serving_endpoints.get(endpoint_name)

print(f"⏰ Status Check - {datetime.datetime.now().strftime('%H:%M:%S')}")
print("=" * 60)
print(f"Ready State: {endpoint.state.ready}")
print(f"Config Update: {endpoint.state.config_update}")

# Check config update first (deployment status)
if str(endpoint.state.config_update) == "EndpointStateConfigUpdate.IN_PROGRESS":
    print("\n⏳ Still deploying... (this is normal)")
    print("   Run this cell again in 2-3 minutes to check progress.")
    print("\n💡 Typical deployment time: 5-15 minutes")
elif str(endpoint.state.ready) == "EndpointStateReady.READY":
    print("\n🎉 ENDPOINT IS READY!")
    print("\n✅ Next steps:")
    print("   1. Test it by running cell 25 (Test the Deployed Endpoint)")
    print("   2. Your Vercel app should now work!")
    print("\n📱 Vercel Environment Variables:")
    print(f"   DATABRICKS_HOST=dbc-ed0aa653-24ab.cloud.databricks.com")
    print(f"   DATABRICKS_TOKEN=<your_token>")
    print(f"   ENDPOINT_NAME={endpoint_name}")
else:
    print(f"\n⚠️  Unexpected state. Check UI: /ml/endpoints/{endpoint_name}")

In [0]:
# Check for detailed deployment errors and pending config
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
endpoint_name = "movie-rag-chatbot-endpoint"

endpoint = w.serving_endpoints.get(endpoint_name)

print("🔍 Detailed Deployment Status")
print("=" * 60)

print(f"\n📊 Current State:")
print(f"   Ready: {endpoint.state.ready}")
print(f"   Config Update: {endpoint.state.config_update}")

if hasattr(endpoint.state, 'ready_message') and endpoint.state.ready_message:
    print(f"   Message: {endpoint.state.ready_message}")

# Check current served models
if endpoint.config and endpoint.config.served_entities:
    print(f"\n✅ Current Deployed Model:")
    for entity in endpoint.config.served_entities:
        print(f"   • {entity.entity_name}")
        print(f"     Version: {entity.entity_version}")
        print(f"     Workload: {entity.workload_size}")

# Check what's being deployed (pending config)
if endpoint.pending_config:
    print(f"\n⏳ Pending Deployment (IN_PROGRESS):")
    if endpoint.pending_config.served_entities:
        for entity in endpoint.pending_config.served_entities:
            print(f"   • {entity.entity_name}")
            print(f"     Version: {entity.entity_version}")
            print(f"     Workload: {entity.workload_size}")
    print(f"\n   ⚠️ This update has been IN_PROGRESS for a long time.")
else:
    print(f"\n✅ No pending config changes")

print(f"\n💡 Recommendations:")
print(f"   1. Check the endpoint UI for detailed logs/errors:")
print(f"      /ml/endpoints/{endpoint_name}")
print(f"\n   2. If stuck >60 min with no errors shown:")
print(f"      Run cell 24 to force restart (delete & recreate)")
print(f"\n   3. Common causes of slow deployment:")
print(f"      • Large model loading dependencies")
print(f"      • Vector search index synchronization")
print(f"      • Resource availability issues")

In [0]:
# Test deployed Model Serving endpoint with sample questions
import requests
import json

# Target endpoint for testing
endpoint_name = "movie-rag-chatbot-endpoint"

print(f"Testing endpoint: {endpoint_name}")
print("=" * 60)

# Get workspace URL and token
workspace_url = spark.conf.get("spark.databricks.workspaceUrl")
token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

# Prepare request
url = f"https://{workspace_url}/serving-endpoints/{endpoint_name}/invocations"
headers = {
    "Authorization": f"Bearer {token}",
    "Content-Type": "application/json"
}

# Test questions
test_questions = [
    "Can you recommend a good action movie?",
    "What are some good comedy films?"
]

for question in test_questions:
    print(f"\nQuestion: {question}")
    
    payload = {
        "dataframe_records": [{"question": question}]
    }
    
    try:
        response = requests.post(url, headers=headers, json=payload)
        
        if response.status_code == 200:
            result = response.json()
            answer = result['predictions'][0]
            print(f"Answer: {answer}")
        else:
            print(f"❌ Error: {response.status_code}")
            print(response.text)
    except Exception as e:
        print(f"❌ Error: {str(e)}")
    
    print("-" * 60)

print("\n✓ Endpoint test complete!")
print(f"\n📖 REST API Usage:")
print(f"   URL: https://{workspace_url}/serving-endpoints/{endpoint_name}/invocations")
print(f"   Method: POST")
print(f"   Headers: Authorization: Bearer <token>")
print(f"   Body: {{\"dataframe_records\": [{{\"question\": \"your question\"}}]}}")

In [0]:
# Wait for the endpoint to be ready and test it
from databricks.sdk import WorkspaceClient
import time

w = WorkspaceClient()
endpoint_name = "movie-rag-chatbot-endpoint"

print(f"⏳ Waiting for endpoint to be ready...")
print("=" * 60)

try:
    # Wait for endpoint to be ready (max 15 minutes)
    endpoint = w.serving_endpoints.wait_get_serving_endpoint_not_updating(
        name=endpoint_name,
        timeout=datetime.timedelta(minutes=15)
    )
    
    print(f"\n✅ Endpoint is now READY!")
    print(f"   State: {endpoint.state.ready}")
    
    # Test the endpoint
    print(f"\n🧪 Testing endpoint...")
    
    response = w.serving_endpoints.query(
        name=endpoint_name,
        dataframe_records=[{"question": "Can you recommend a good action movie?"}]
    )
    
    answer = response.predictions[0]
    print(f"\n✅ Test successful!")
    print(f"\nQuestion: Can you recommend a good action movie?")
    print(f"Answer: {answer}")
    
    print("\n" + "=" * 60)
    print("\n🎉 Your endpoint is ready for Vercel!")
    print("\n📝 Vercel Environment Variables:")
    print("=" * 60)
    print(f"DATABRICKS_HOST=dbc-ed0aa653-24ab.cloud.databricks.com")
    print(f"DATABRICKS_TOKEN=<your_active_token>")
    print(f"ENDPOINT_NAME={endpoint_name}")
    print("\n⚠️  Important: DATABRICKS_HOST should NOT include 'https://'")
    print("\n✅ Your Vercel app should now work!")
    
except Exception as e:
    print(f"\n❌ Error or timeout: {e}")
    print(f"\nThe endpoint may still be deploying.")
    print(f"Check status in Databricks UI: /ml/endpoints/{endpoint_name}")

In [0]:
# Check if the model serving endpoint is ready and responding
from databricks.sdk import WorkspaceClient
import json

w = WorkspaceClient()

endpoint_name = "movie-rag-chatbot-endpoint"

print(f"🔍 Checking endpoint: {endpoint_name}")
print("=" * 60)

try:
    # Get endpoint details
    endpoint = w.serving_endpoints.get(endpoint_name)
    
    print(f"\nℹ️ Endpoint Status:")
    print(f"   Name: {endpoint.name}")
    print(f"   State: {endpoint.state.ready}")
    print(f"   Config State: {endpoint.state.config_update}")
    
    # Check served entities (models)
    if endpoint.config and endpoint.config.served_entities:
        print(f"\n📦 Served Models:")
        for entity in endpoint.config.served_entities:
            print(f"   • {entity.entity_name} (version {entity.entity_version})")
            print(f"     Scale to zero: {entity.scale_to_zero_enabled}")
            print(f"     Workload size: {entity.workload_size}")
    
    # Check if endpoint is ready
    if endpoint.state.ready == "READY":
        print(f"\n✅ Endpoint is READY and should be working!")
        
        # Try a test request
        print(f"\n🧪 Testing endpoint with a sample question...")
        
        try:
            response = w.serving_endpoints.query(
                name=endpoint_name,
                dataframe_records=[{"question": "Can you recommend a good action movie?"}]
            )
            
            print(f"\n✅ Test request successful!")
            print(f"   Response: {response.predictions[0][:100]}...")
            print(f"\n💡 The endpoint is working. Your Vercel error might be:")
            print(f"   1. Wrong endpoint URL in Vercel")
            print(f"   2. Wrong DATABRICKS_HOST format (should NOT include https://)")
            print(f"   3. Token not properly set in Vercel environment variables")
            print(f"   4. CORS or network issues")
            
        except Exception as test_error:
            print(f"\n❌ Test request failed: {test_error}")
            print(f"\n🔧 Possible causes:")
            print(f"   1. Model dependencies not loaded")
            print(f"   2. Vector search index not synced")
            print(f"   3. Endpoint needs to restart")
            
    elif endpoint.state.ready == "NOT_READY":
        print(f"\n⚠️  Endpoint is NOT_READY")
        print(f"   This is normal after deployment. Wait 5-10 minutes.")
        
    elif endpoint.state.ready == "UPDATE_IN_PROGRESS":
        print(f"\n⏳ Endpoint is updating...")
        print(f"   Wait for the update to complete before testing.")
        
    else:
        print(f"\n❌ Endpoint state: {endpoint.state.ready}")
        print(f"   Check the endpoint in the Databricks UI for errors.")
    
    print(f"\n\n🔗 Vercel Environment Variables Should Be:")
    print("=" * 60)
    print(f"DATABRICKS_HOST=dbc-ed0aa653-24ab.cloud.databricks.com")
    print(f"DATABRICKS_TOKEN=<your_token_here>")
    print(f"ENDPOINT_NAME={endpoint_name}")
    print("\n⚠️  Note: DATABRICKS_HOST should NOT include 'https://'")
    
except Exception as e:
    print(f"\n❌ Error: {e}")
    
    if "NOT_FOUND" in str(e) or "does not exist" in str(e):
        print(f"\n🚫 Endpoint '{endpoint_name}' does not exist!")
        print(f"\n🔧 Solution:")
        print(f"   1. Run cell 23 to create the endpoint")
        print(f"   2. Wait 5-10 minutes for it to be ready")
        print(f"   3. Then update Vercel environment variables")
    else:
        print(f"\nUnexpected error. Check Databricks UI for endpoint status.")

In [0]:
# Check Databricks token expiration and generate new token for Vercel
from databricks.sdk import WorkspaceClient
import datetime

print("🔍 Checking your Databricks tokens...")
print("=" * 60)

w = WorkspaceClient()

try:
    # List all tokens for the current user
    tokens = w.token_management.list()
    
    print("\n📋 Your Databricks Personal Access Tokens:\n")
    
    active_tokens = []
    expired_tokens = []
    
    for token in tokens:
        token_id = token.token_id
        comment = token.comment or "(no comment)"
        creation_time = datetime.datetime.fromtimestamp(token.creation_time / 1000)
        
        if token.expiry_time:
            expiry_time = datetime.datetime.fromtimestamp(token.expiry_time / 1000)
            time_left = expiry_time - datetime.datetime.now()
            
            if time_left.total_seconds() > 0:
                days_left = time_left.days
                status = "✅ ACTIVE"
                active_tokens.append((token_id, comment, expiry_time, days_left))
            else:
                status = "❌ EXPIRED"
                expired_tokens.append((token_id, comment, expiry_time))
            
            print(f"{status} | {comment[:30]}")
            print(f"   ID: {token_id}")
            print(f"   Created: {creation_time.strftime('%Y-%m-%d %H:%M')}")
            print(f"   Expires: {expiry_time.strftime('%Y-%m-%d %H:%M')}")
            
            if status == "✅ ACTIVE":
                print(f"   Time left: {days_left} days")
                if days_left < 7:
                    print(f"   ⚠️  WARNING: Expires soon!")
        else:
            print(f"✅ ACTIVE (no expiration) | {comment[:30]}")
            print(f"   ID: {token_id}")
            print(f"   Created: {creation_time.strftime('%Y-%m-%d %H:%M')}")
            active_tokens.append((token_id, comment, None, None))
        
        print("-" * 60)
    
    print("\n📊 Summary:")
    print(f"   Active tokens: {len(active_tokens)}")
    print(f"   Expired tokens: {len(expired_tokens)}")
    
    if expired_tokens:
        print("\n⚠️  You have expired tokens! This is likely causing your Vercel errors.")
        print("\n🔧 Solution:")
        print("   1. Generate a new token (see instructions below)")
        print("   2. Update DATABRICKS_TOKEN in Vercel environment variables")
        print("   3. Redeploy your Vercel app")
    
    if any(days_left and days_left < 7 for _, _, _, days_left in active_tokens if days_left):
        print("\n⚠️  Some tokens expire soon. Consider creating a long-lived token.")
    
    print("\n" + "=" * 60)
    print("\n📝 To create a new token for Vercel:")
    print("=" * 60)
    print("\n1. Go to: User Settings → Developer → Access Tokens")
    print("2. Click 'Generate New Token'")
    print("3. Settings:")
    print("   • Comment: 'Vercel Production'")
    print("   • Lifetime: 90 days (or longer for production)")
    print("   • Permissions: Can access all APIs")
    print("4. Copy the token immediately (you won't see it again!)")
    print("5. Update in Vercel:")
    print("   • Settings → Environment Variables")
    print("   • Update DATABRICKS_TOKEN with the new value")
    print("   • Redeploy your application")
    print("\n💡 For production: Use a Service Principal instead of a personal token")
    print("   Service Principals don't expire and are better for automation")
    
except Exception as e:
    print(f"❌ Error checking tokens: {e}")
    print("\nYou may not have permission to list tokens.")
    print("Try generating a new token manually in User Settings → Developer.")

In [0]:
# Keep-alive / wake-up script: ping the model serving endpoint every 5 days to prevent scale-to-zero
# This cell is designed to be scheduled as a recurring Databricks Job (every 5 days).
# The endpoint has scale_to_zero_enabled=True, so it goes idle when not used.
# This script:
#   1. Checks the endpoint status
#   2. If NOT_READY, waits for it to scale up (up to 10 min)
#   3. Sends a warm-up query to keep it active
#   4. Logs the result with a timestamp

from databricks.sdk import WorkspaceClient
import json
import datetime
import time

KEEPALIVE_ENDPOINT_NAME = "movie-rag-chatbot-endpoint"
WARMUP_QUERY = "Can you recommend a good action movie?"
MAX_WAIT_MINUTES = 10

def get_endpoint_status(w):
    """Get the current state of the serving endpoint"""
    try:
        endpoint = w.serving_endpoints.get(KEEPALIVE_ENDPOINT_NAME)
        return endpoint.state.ready if endpoint.state else "UNKNOWN"
    except Exception as e:
        print(f"  WARNING: Could not get endpoint status: {e}")
        return "NOT_FOUND"

def wait_for_endpoint_ready(w, max_minutes=MAX_WAIT_MINUTES):
    """Wait for the endpoint to become READY (it may be scaling up from zero)"""
    status = get_endpoint_status(w)
    if status == "READY":
        return True
    if status == "NOT_FOUND":
        return False
    print(f"  Endpoint is {status}, waiting for it to scale up (up to {max_minutes} min)...")
    start = time.time()
    while time.time() - start < max_minutes * 60:
        time.sleep(30)
        status = get_endpoint_status(w)
        elapsed = int((time.time() - start) / 60)
        print(f"  [{elapsed}m] Status: {status}")
        if status == "READY":
            return True
        if status == "NOT_FOUND":
            return False
    print(f"  Timed out after {max_minutes} minutes. Endpoint still {status}.")
    return False

def keep_endpoint_alive():
    """Wake up and ping the model serving endpoint to keep it active"""
    w = WorkspaceClient()
    now = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print(f"\n{'='*60}")
    print(f"[{now}] Keep-alive wake-up script starting...")
    print(f"  Endpoint: {KEEPALIVE_ENDPOINT_NAME}")
    print(f"{'='*60}")
    status = get_endpoint_status(w)
    print(f"\n  Initial endpoint status: {status}")
    if status == "NOT_FOUND":
        print(f"  Endpoint '{KEEPALIVE_ENDPOINT_NAME}' not found.")
        print("  Make sure the serving endpoint has been created (see earlier cells).")
        return False
    if status != "READY":
        print(f"\n  Endpoint is {status} -- waiting for scale-up...")
        if not wait_for_endpoint_ready(w):
            print(f"  Endpoint did not become READY in time.")
            return False
        print(f"  Endpoint is now READY!")
    host = w.config.host.rstrip("/")
    url = f"{host}/serving-endpoints/{KEEPALIVE_ENDPOINT_NAME}/invocations"
    headers = w.config.authenticate()
    headers["Content-Type"] = "application/json"
    payload = json.dumps({"inputs": [{"question": WARMUP_QUERY}]})
    print(f"\n  Sending warm-up query: '{WARMUP_QUERY}'")
    try:
        import requests
        response = requests.post(url, headers=headers, data=payload, timeout=60)
        if response.status_code == 200:
            result = response.json()
            try:
                preds = result.get("predictions", [])
                answer = preds[0] if preds else "(no prediction returned)"
            except Exception:
                answer = str(result)[:200]
            print(f"  Keep-alive ping succeeded (HTTP 200)")
            print(f"  Response preview: {str(answer)[:150]}...")
            try:
                log_conversation_to_delta("keepalive_schedule", WARMUP_QUERY, str(answer)[:500])
                print("  Logged to Delta analytics table")
            except Exception as e:
                print(f"  Could not log to Delta: {e}")
            return True
        else:
            print(f"  Keep-alive ping failed: HTTP {response.status_code}")
            print(f"  Response: {response.text[:300]}")
            return False
    except Exception as e:
        print(f"  Request failed: {e}")
        return False

# -- Run the keep-alive --
success = keep_endpoint_alive()
now = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"\n{'='*60}")
print(f"[{now}] Keep-alive result: {'SUCCESS' if success else 'FAILED'}")
print(f"{'='*60}")
print(f"\nThis script runs every 5 days via a scheduled Databricks Job.")
print(f"  Next scheduled run: 5 days from now.")
print(f"  View job runs in the Jobs sidebar.")